# Method of Lines and Runge-Kutta

## Authors: Zach Etienne and Brandon Clark

This notebook adapts the NRPy Method-of-Lines tutorial material and the
current BHaH MoL time-stepping source.

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook shows how an RK4 Butcher table maps to NRPy stage storage and
how NRPy registers a generated C driver for one Method-of-Lines time step.

**Notebook Status:** Validated

**Validation Notes:** The validation sections confirm the trusted RK4 table,
stage-storage names, and generated Method-of-Lines schedule containing four
RHS calls and four RK launcher calls.

Navigation: [Index](../index.ipynb) |
Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) |
Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)

# Table of Contents

1. [Required Reading and Source Links](#Required-Reading-and-Source-Links)
1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Step 1](#Step-1:-State-the-Method-of-Lines-Time-Problem): State the time-step problem.
1. [Step 2](#Step-2:-Inspect-the-RK4-Butcher-Table): Inspect the RK4 table.
1. [Step 3](#Step-3:-Validate-the-RK4-Table-and-Storage-Names): Validate table metadata.
1. [Step 4](#Step-4:-Map-RK4-Rows-to-NRPy-Stage-Storage): Map storage.
1. [Step 5](#Step-5:-Register-the-RK-Step-Forward-C-Function): Register the C function.
1. [Step 6](#Step-6:-Inspect-the-Generated-RK-Stage-Schedule): Inspect generated evidence.
1. [Validation Check](#Validation-Check): Validate generated schedule.
1. [What next?](#What-next?)

# Required Reading and Source Links
### [Back to [top](#Table-of-Contents)]

Required reading:

- [Wave Equation with NumPy](wave_equation_with_numpy.ipynb): shows the
  PDE-to-ODE context that motivates Method-of-Lines time stepping.
- [C Function tutorial](../1-intro/c_function.ipynb): useful background for
  the CFunction registry. This notebook also defines the CFunction pieces it
  uses before registration.

Installed source modules used by this lesson:

- `nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary`:
  trusted RK coefficients and stage-storage metadata.
- `nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time`:
  BHaH registration helper that assembles the generated C driver.
- `nrpy.infrastructures.BHaH.MoLtimestepping.rk_substep`: registered
  stage-kernel metadata.
- `nrpy.c_function`: generated C function object, prototype, body, and
  full-function assembly.

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Method of Lines:** A workflow that first discretizes spatial derivatives,
  then advances the resulting ordinary differential equation in time.
- **Semi-discrete system:** The ODE `dU/dt = R(U)` produced after spatial
  discretization.
- **`U_n`:** The vector of all evolved grid fields at time `t_n`.
- **Right-hand side:** The function `R(U)` that gives the time derivative of
  the evolved state.
- **RK4:** The classical four-stage Runge-Kutta method.
- **Butcher table:** The table of stage times and weights defining a
  Runge-Kutta method. See the
  [Runge-Kutta overview](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods).
- **Stage storage:** Temporary grid-field arrays used during intermediate RK
  calculations.
- **C function:** A generated C routine with a return type, name, parameter
  list, and body.
- **Function prototype:** A C declaration that lists the return type, name,
  and parameter list.
- **Registration:** Storing a generated C function or RK substep in NRPy's
  registries so later infrastructure can write source files.

# Step 1: State the Method-of-Lines Time Problem
### [Back to [top](#Table-of-Contents)]

A spatial discretization turns fields on a grid into one state vector `U(t)`.
The semi-discrete system is

$$
\frac{dU}{dt} = R(U).
$$

One time step starts from known data `U_n` at time `t_n` and produces
`U_{n+1}`. This notebook does not solve a full PDE initial-value problem; it
shows how NRPy represents the RK4 driver that would advance such a system.

RK4 asks for four evaluations of `R(U)`, combines them with Butcher-table
weights, and stores intermediate states in named gridfunction arrays. NRPy
then maps those stages to generated C launcher calls.

The generated C function has the usual C shape:

```c
return_type function_name(parameter_list) {
    local_declarations;
    assignments_or_algorithm;
}
```

In NRPy, the pieces inspected here are the function prototype, parameter
list, full generated body, registered function dictionary entry, and the
registered RK substep helper functions.

This source-provenance check must resolve NRPy through the pip-installed
package. It raises immediately if a local cloned `nrpy/` directory shadows the
installed package.

In [1]:
from pathlib import Path
import importlib.util


NOTEBOOK_DIR = Path.cwd().resolve()
FORBIDDEN_SOURCE_ROOTS = [NOTEBOOK_DIR.parent / "nrpy"]


def installed_source_path(module_name):
    spec = importlib.util.find_spec(module_name)
    if spec is None or spec.origin is None:
        raise RuntimeError(f"Could not locate installed module: {module_name}")
    path = Path(spec.origin).resolve()
    for root in FORBIDDEN_SOURCE_ROOTS:
        if not root.exists():
            continue
        try:
            path.relative_to(root.resolve())
        except ValueError:
            continue
        raise RuntimeError(f"{module_name} resolved to cloned source: {path}")
    return path


source_modules = [
    "nrpy.c_function",
    "nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time",
    "nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary",
    "nrpy.infrastructures.BHaH.MoLtimestepping.rk_substep",
]
print("installed NRPy source paths:")
for module_name in source_modules:
    print(module_name, "->", installed_source_path(module_name))

installed NRPy source paths:
nrpy.c_function -> /virt/lib/python3.12/site-packages/nrpy/c_function.py


nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time -> /virt/lib/python3.12/site-packages/nrpy/infrastructures/BHaH/MoLtimestepping/MoL_step_forward_in_time.py
nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary -> /virt/lib/python3.12/site-packages/nrpy/infrastructures/BHaH/MoLtimestepping/rk_butcher_table_dictionary.py
nrpy.infrastructures.BHaH.MoLtimestepping.rk_substep -> /virt/lib/python3.12/site-packages/nrpy/infrastructures/BHaH/MoLtimestepping/rk_substep.py


These setup imports expose the RK table utilities, BHaH registration helper,
CFunction registry, and SymPy rational numbers used for exact table checks.

In [2]:
import sympy as sp

import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import (
    register_CFunction_MoL_step_forward_in_time,
)
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)

# Step 2: Inspect the RK4 Butcher Table
### [Back to [top](#Table-of-Contents)]

The table rows list stage times and stage weights. The final row lists the
weights used to combine the four stages into `U_{n+1}`.

Inspect:

- four stage rows;
- final weights `1/6, 1/3, 1/3, 1/6`;
- diagonal status;
- intermediate stage-storage names.

In [3]:
butcher_tables = generate_Butcher_tables()
rk4_table, rk4_order = butcher_tables["RK4"]
rk4_stage_rows = rk4_table[:-1]
rk4_weight_row = rk4_table[-1]
rk4_storage_names = intermediate_stage_gf_names_list(butcher_tables, "RK4")

formatted_rows = []
for stage, row in enumerate(rk4_stage_rows, start=1):
    c_value = row[0]
    a_values = list(row[1:]) + [""] * (4 - len(row[1:]))
    formatted_rows.append((stage, c_value, *a_values[:4]))

print("stage c a1 a2 a3 a4")
for formatted_row in formatted_rows:
    print(*formatted_row)
print("b weights:", rk4_weight_row[1:])
print("RK4 order:", rk4_order)
print("is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print("intermediate storage names:", rk4_storage_names)

stage c a1 a2 a3 a4
1 0    
2 1/2 1/2   
3 1/2 0 1/2  
4 1 0 0 1 
b weights: [1/6, 1/3, 1/3, 1/6]
RK4 order: 4
is diagonal: True
intermediate storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']


The storage names are NRPy's bridge between the mathematical RK4 table and
generated infrastructure. The next section validates the table before any
CFunction registration.

# Step 3: Validate the RK4 Table and Storage Names
### [Back to [top](#Table-of-Contents)]

The trusted result is the classical RK4 table stored in NRPy's table
dictionary. The newly inspected metadata are the table returned by this run
and the stage-storage names used by generated BHaH infrastructure.

In [4]:
validate(butcher_tables, ["RK4"], verbose=False)
expected_rk4_table = [
    [0],
    [sp.Rational(1, 2), sp.Rational(1, 2)],
    [sp.Rational(1, 2), 0, sp.Rational(1, 2)],
    [1, 0, 0, 1],
    [
        "",
        sp.Rational(1, 6),
        sp.Rational(1, 3),
        sp.Rational(1, 3),
        sp.Rational(1, 6),
    ],
]
expected_storage_names = [
    "y_nplus1_running_total_gfs",
    "k_odd_gfs",
    "k_even_gfs",
]
print("validated RK4 order:", rk4_order)
print("stage rows:", len(rk4_stage_rows))
print("storage names:", rk4_storage_names)
if rk4_order != 4:
    raise RuntimeError("Expected RK4 to report order 4.")
if rk4_table != expected_rk4_table:
    raise RuntimeError("Expected the classical RK4 Butcher coefficients.")
if rk4_storage_names != expected_storage_names:
    raise RuntimeError("Expected RK4 to use the standard NRPy stage storage.")
print("PASS: RK4 table and storage names match trusted metadata")

RK4: PASSED VALIDATION
validated RK4 order: 4
stage rows: 4
storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']
PASS: RK4 table and storage names match trusted metadata


# Step 4: Map RK4 Rows to NRPy Stage Storage
### [Back to [top](#Table-of-Contents)]

The Butcher table gives the stage weights. NRPy also needs named arrays that
hold the current fields, temporary stage data, and running total. The next
complete storage map is the catalog learners should use when reading the
generated stage schedule.

In [5]:
stage_storage_rows = [
    ("current fields", "y_n_gfs", "input state U_n"),
    ("running total", "y_nplus1_running_total_gfs", "weighted sum for U_{n+1}"),
    ("odd stages", "k_odd_gfs", "stage 1 and stage 3 data"),
    ("even stages", "k_even_gfs", "stage 2 and stage 4 data"),
]
print("role | storage name | purpose")
for role, storage_name, purpose in stage_storage_rows:
    print(role, "|", storage_name, "|", purpose)

role | storage name | purpose
current fields | y_n_gfs | input state U_n
running total | y_nplus1_running_total_gfs | weighted sum for U_{n+1}
odd stages | k_odd_gfs | stage 1 and stage 3 data
even stages | k_even_gfs | stage 2 and stage 4 data


# Step 5: Register the RK Step-Forward C Function
### [Back to [top](#Table-of-Contents)]

The BHaH registration helper assembles the Method-of-Lines C driver and
records the generated function in `cfc.CFunction_dict`. The helper also
registers RK substep metadata in `BHaH.MoLtimestepping.rk_substep`.

Inspect the complete prototype, the registered substep names, and the writes
performed by each substep.

In [6]:
cfc.CFunction_dict.pop("MoL_step_forward_in_time", None)
for name in ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]:
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(name, None)

register_CFunction_MoL_step_forward_in_time(
    butcher_tables,
    "RK4",
    rhs_string="rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);",
)

registered = cfc.CFunction_dict["MoL_step_forward_in_time"]
registered_substeps = BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict

print("registered MoL function names:")
for name in sorted(name for name in cfc.CFunction_dict if "MoL" in name or "mol" in name):
    print(name)
print("complete C function prototype:")
print(registered.function_prototype)
print("registered substep names:", sorted(registered_substeps))
print("stage | launcher | writes")
for stage_name, rk_function in sorted(registered_substeps.items()):
    writes = ", ".join(rk_function.RK_lhs_str_list)
    print(stage_name, "|", rk_function.name, "|", writes)

registered MoL function names:
MoL_step_forward_in_time
complete C function prototype:
void MoL_step_forward_in_time(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
registered substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']
stage | launcher | writes
RK_STEP1 | rk_substep_1__launcher | y_nplus1_running_total_gfs[i], k_odd_gfs[i]
RK_STEP2 | rk_substep_2__launcher | y_nplus1_running_total_gfs[i], k_even_gfs[i]
RK_STEP3 | rk_substep_3__launcher | y_nplus1_running_total_gfs[i], k_odd_gfs[i]
RK_STEP4 | rk_substep_4__launcher | y_n_gfs[i]


# Step 6: Inspect the Generated RK Stage Schedule
### [Back to [top](#Table-of-Contents)]

The complete required schedule evidence is the bounded table below: every RK
stage marker, RHS call, and launcher call parsed from the generated driver.
The full generated driver also contains support code; the representative
generated kernel printed afterward is the complete first substep function.

In [7]:
full_function = registered.full_function
schedule_rows = []
current_stage = None
for line in full_function.splitlines():
    stripped = line.strip()
    if stripped.startswith("// -={ START k"):
        current_stage = stripped
        schedule_rows.append((current_stage, "stage marker", stripped))
    elif current_stage is not None and stripped.startswith("rhs_eval("):
        schedule_rows.append((current_stage, "RHS call", stripped))
    elif (
        current_stage is not None
        and stripped.startswith("rk_substep_")
        and stripped.endswith(";")
    ):
        schedule_rows.append((current_stage, "launcher call", stripped))

print("complete generated RK stage schedule:")
print("stage marker | kind | generated line")
for stage_marker, kind, generated_line in schedule_rows:
    print(stage_marker, "|", kind, "|", generated_line)

start = full_function.index("static void rk_substep_1_host")
end_marker = "} // END FUNCTION: rk_substep_1_host"
end = full_function.index(end_marker, start) + len(end_marker)
print("complete representative generated first RK substep function:")
print(full_function[start:end])

complete generated RK stage schedule:
stage marker | kind | generated line
// -={ START k1 substep }=- | stage marker | // -={ START k1 substep }=-
// -={ START k1 substep }=- | RHS call | rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
// -={ START k1 substep }=- | launcher call | rk_substep_1__launcher(params, k_odd_gfs, y_n_gfs, y_nplus1_running_total_gfs, commondata->dt);
// -={ START k2 substep }=- | stage marker | // -={ START k2 substep }=-
// -={ START k2 substep }=- | RHS call | rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
// -={ START k2 substep }=- | launcher call | rk_substep_2__launcher(params, k_even_gfs, y_nplus1_running_total_gfs, y_n_gfs, commondata->dt);
// -={ START k3 substep }=- | stage marker | // -={ START k3 substep }=-
// -={ START k3 substep }=- | RHS call | rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
// -={ START k3 substep }=- | launcher call | rk_substep_3__launcher(params, k_odd_gf

# Validation Check
### [Back to [top](#Table-of-Contents)]

This check compares the newly generated schedule and substep registry against
the trusted RK4 table metadata. It uses counts and registered names rather
than whitespace-sensitive formatting.

In [8]:
expected_substep_names = ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]
actual_substep_names = sorted(registered_substeps)
rhs_call_count = sum(1 for _, kind, _ in schedule_rows if kind == "RHS call")
launcher_call_count = sum(1 for _, kind, _ in schedule_rows if kind == "launcher call")
stage_marker_count = sum(1 for _, kind, _ in schedule_rows if kind == "stage marker")

print("trusted source: RK4 Butcher table and NRPy registration metadata")
print("new artifact: registered Method-of-Lines function and substep dictionary")
print("expected substep names:", expected_substep_names)
print("actual substep names:", actual_substep_names)
print("stage marker count:", stage_marker_count)
print("RHS call count:", rhs_call_count)
print("launcher call count:", launcher_call_count)
if actual_substep_names != expected_substep_names:
    raise RuntimeError("Expected four registered RK4 substeps.")
if stage_marker_count != 4:
    raise RuntimeError("Expected four generated RK4 stage markers.")
if rhs_call_count != 4:
    raise RuntimeError("Expected four generated RHS calls.")
if launcher_call_count != 4:
    raise RuntimeError("Expected four generated RK launcher calls.")
print("PASS: generated RK4 schedule matches trusted RK4 metadata")

trusted source: RK4 Butcher table and NRPy registration metadata
new artifact: registered Method-of-Lines function and substep dictionary
expected substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']
actual substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']
stage marker count: 4
RHS call count: 4
launcher call count: 4
PASS: generated RK4 schedule matches trusted RK4 metadata


The validation confirms both layers of evidence: the RK4 table metadata and
the generated C schedule that calls the RHS and one launcher for each stage.

# What next?
### [Back to [top](#Table-of-Contents)]

- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)